# Object Tracking

### Detection vs Tracking

Detection answers:
- what objects are in this frame?

Tracking answers:
- what objects are in this frame?
- are they the same objects from last frame?

Without tracking:
- boxes appear and disappear every frame
- no memory of what was where
- same person counted multiple times

With tracking:
- each object gets a persistent ID
- ID survives across frames
- ID survives brief disappearances

### How Tracking Works — ID Matching

Core idea:
- object at `(300, 200)` in frame 32
- closest box to `(300, 200)` in frame 33 → same object

Matching is done using IoU:
- high IoU between new box and previous box → same object, keep ID
- no IoU match found → new object, assign new ID
- previous ID not matched → object left frame, remember briefly

```
Frame 32:   ID#1 at (300, 200)    ID#2 at (500, 300)                    
                ↓                       ↓                               
Frame 33:   ID#1 at (310, 210)    ID#2 at (490, 310)   ← matched by IoU 
```

### SORT — Simple Online and Realtime Tracking

The algorithm behind basic tracking:
- matches boxes frame to frame using IoU
- position only — no appearance information
- fast and lightweight

Limitation:
- two objects cross paths → IDs can swap
- long occlusions → ID lost

This is solved by DeepSORT, covered in the next module.

### box.id — The Persistent ID

`model.track()` returns the same result as `model()` with one addition:
- `box.id` — integer ID assigned to this object

Important:
- `box.id` can be `None` for the first 1-2 frames a new object appears
- tracker needs a few frames to confirm it's a real detection, not noise
- always guard against `None` before using the ID

```python
if box.id is not None:
    track_id = int(box.id.item())
```

### Setup

In [1]:
from ultralytics import YOLO
import cv2
import time

### Constants

In [2]:
DETECT_EVERY   = 5
CONF_THRESHOLD = 0.5

### Load Model

In [3]:
model = YOLO('yolov8n.pt')

### Tracking Function

In [4]:
def track(frame):

    results = model.track(frame, conf=CONF_THRESHOLD, persist=True, verbose=False)
    result  = results[0]

    detections = []

    for box in result.boxes:

        if box.id is None:
            continue

        class_id   = int(box.cls.item())
        label      = model.names[class_id]
        confidence = box.conf.item()
        track_id   = int(box.id.item())
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        detections.append((label, confidence, track_id, (x1, y1, x2, y2)))

    return detections

#### `model.track()` vs `model()`
- same as `model()` but enables tracking
- assigns persistent IDs across frames

#### `persist=True`
- tells the tracker to remember IDs between calls
- without this — tracker resets every frame, IDs restart from 0

#### `if box.id is None: continue`
- skips boxes that haven't been confirmed yet
- prevents crash on first appearance of a new object

### Draw Function

In [5]:
def draw(frame, detections):

    for label, confidence, track_id, (x1, y1, x2, y2) in detections:

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text = f"{label} {confidence:.2f} #{track_id}"
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame

Label format:

```
person 0.91 #3
car    0.87 #1
```

### Webcam Loop

In [6]:
cap = cv2.VideoCapture(0)

stored_detections = []
seen_ids          = set()
frame_count       = 0
fps               = 0
prev_time         = time.time()
display_fps       = "FPS: --"

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Detection
    if frame_count % DETECT_EVERY == 0:
        stored_detections = track(frame)

        for label, confidence, track_id, box in stored_detections:
            seen_ids.add(track_id)

    # Draw
    frame = draw(frame, stored_detections)

    # FPS
    new_time = time.time()
    fps = 0.9 * fps + 0.1 * (1 / (new_time - prev_time))
    prev_time = new_time

    if frame_count % DETECT_EVERY == 0:
        display_fps = f"FPS: {int(fps)}"

    if not stored_detections:
        display_fps = "Idle"

    # Display
    cv2.putText(frame, display_fps, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f"Total seen: {len(seen_ids)}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### `seen_ids = set()`
- stores every unique ID ever detected
- set automatically ignores duplicates
- `len(seen_ids)` = total unique objects seen since start

#### `seen_ids.add(track_id)`
- only updated on detection frames, not skipped frames
- no point adding IDs from stale stored detections

### Summary

| Concept | What It Does |
| ------- | ------------ |
| `model.track()` | Detection + persistent ID assignment |
| `persist=True` | Keeps tracker state between frames |
| `box.id` | Persistent integer ID for this object |
| `box.id is None` | Object just appeared, not confirmed yet |
| `seen_ids` | Set of all unique IDs ever detected |
| SORT | IoU-based matching — position only |

Limitation of SORT:
- two objects cross paths → IDs can swap
- long occlusion → ID lost

Next up — DeepSORT adds appearance embeddings on top of position matching to handle these cases.